# REINVENT: внешний benchmark

Запуски на 25% и 100% `curation v2`. Неподдерживаемые prior токены сохраняются в таблице coverage.

In [ ]:
%pip install -q "mmpdb>=2.1,<3" tensorboard molvs xxhash python-dotenv apted pumas polars rdkit
%pip install -q --no-deps "git+https://github.com/MolecularAI/REINVENT4.git@v4.8"

import subprocess, sys
subprocess.run([sys.executable, "-c", "from mmpdblib.do_fragment import fragment_command; print('mmpdb OK')"], check=True)

In [ ]:
import base64
import gzip
import json
import math
import random
import shutil
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


TRAIN_B64 = """H4sIAAAAAAAC/8VbuY7bMBDt/SUkAhUkndKNB3Bpqw9SqV1s/r8L75sUObSTBWzTksVjzveGXDjYIf8I3MnB5ccXzd+4fmve+mYXAAKUyA7YQR40eTH1ql0FYgY+GH0yUH/soq8QPRH1x6lrPMtLVD4pB2ZmbKBPAu4WdiXuedmvmons9CCbuSg/hb5g3oR84zuXk1azFdUehX5r3gJOdV/xYHbibhQOZkriuB5ETWInx089xZ80NPiVAgjf2dfdC1YNZZs7AStk6hrRuDuQm5MjUDOLTGP8QRtX7bjAZOMVdLdXVPbSP4IL6F/LFqh39ZI/kZORhmLE2hUqSLn4B31PstOnbmlLnjBEJ6JIHnKEm9Ns9oB40PpVUNrJbST7fgOhZS94KjZQlsSsR8h+vCp8Q7kJcLOwilisqhu33Cgv7ysM3KqnZHVxHexOyNNz8Usyfe3eWO/s5ezkxfSvILZQa01C3lV2KEe/FhO8qlf1quoVhJFk7mFGEY3QIw7rd1d9+UpDgwsTgr7Sxd6JcqbQT80PkOEp1yc09YkwkVTUiYi0eQJHxgU1UxVyONEOoaWqgrVt4uNnEKwKZjL0vCCKd3qtbhEMaPY9NT8T4+/CaXdzDaljxuvWmMdpv37simqSI25JRKlgIUU/YoduadIoGvjTKDxye2nRcCpT+jnjQ1pftmqQa3BSHO4pdtraQE5EuUsGjxwKzjYFPH2a5uvStFlsYrU7ysdPHEN/35zDU4/QZocpIm3H5JU8TUa2g+YN3rY/AyK5T5fw4q20cWK5QtuDoIDHpHkYBY0OdMxXn1gjmYgHNhwk8YDBr/v2u/FY4/Kv/cdvFlB/wF1poHstRDmbxblRso7h0gSudOcJbbi7sY0QreOyAEvsbAIQ+yZr9OgGk6gURjIYECwK29FIcpLm3RyqNRh2aFUaQGFXdr7ocVAgISk7JnmAYV7cw3e4PEDeIg9nUbqtva4KDaNfSMt5koBpdCIxOiiQC3HMgvpGBFxkMuKYdXDnC26UqQ54ln9VxOCNLuSafIRWcYJXRXOeaqfITC2fJIlKaD5amYqV8BhWt7R0waBNnGxWR3biQJJHSzQntAXiHlajfcqadD3vuYzpftxJmGBqNWZZ9smDbdbRZlCBe5zxOlHcAlGsiPaq39q3VPebOMn7gZ6j84PlKkVOZH14A7u3egy2j5J5gZEVSq5z1jb8nXB0OPdvI1aDF0ItBALsqHr5nPP3MauxoD/IUoOIrR2T9zNryFVkJvh9ykmC/5NJ1hNZSebvoRRr46X1CSc4L0kO3RigdCrT/O5Xiy9uoet+Se7PhcxvcVJyafUGfN47MFW+8ZLcSW0PBfJslW+2pO9LeyxHDYhyiKn0VTlcXqW/N324W74rc5mDntj9g7Otha5LnJfJhovRWXFsqnI1jAwG61WjMUczJV9oSkMgfQO065UKlOAbcLxSI8BX4vBPphsGvTpTK9ltEQd+M1wZRiCnJRmc/kKd5USJ/TLKB80rxl2lSVlIlG0X6ZlZMIWsvBdllRlnDOG1XlCpB7KmfRr5IyshZc0Iy6emKxKhWlSw6Zitx21j0+2bDW6N7S7JhS1bClBmwaT6zPu9lX9MzSJl79zydDf7hIljcWKGOtoMvJPpfYFJZfg1EcYHKQxHvziAWUTCjKGXZw8Ccd5E2MWvF/Rb3/sEedji6rQ4ScKIrZzqVGeiecoNq8jSM8Ao6aCV25bC0wkWFiI7EJ+XrK5W8UmLKGLxdH/PRbE0y8ziCHeDtw+acaKVgw8ztCXcvQItvmck5oj5y4xXNDYCayeuopqi8CjK8RZ3XEusn8WqbuCer6CKL3M6Xh7ZgBVQP3Suo5LyNl8/OT86tMKjp4Q5gKbz6tIQel2LKJ1NvdJR13eo8oCbHy6BaHsn2V1BqKe1z4LRdHaIzRRL2w/07vkq7US2P4FxGOFMADrkXmEV2lU2TBCzr26dNE+aFbireuBVx6qyPjLLYCfL38gNlHqNzTHut+bnMqYnVfKJsNzCROhTMZXTywO183mws4qDs8OdLcLJaPdminOOj5VUB6jJwtmkNwDgGSK+WA8M5+SXgRf09hXH4tTgppSmcJ8IBel/NEzwomIrNKzuGdP63k5PKHCh2F6xKfQZZH/uN327Gh0ssqazPfqpHdbOzvxJhlw6T4n/L5liQzU/ST+/yRZN679F2DzmvQs3WYRdO3lSPbUzGpcmYEAWnlaTa01bH3fkAfVMnAPqKOUfpPoSDf8F6H8v7bs2AAA="""
VALIDATION_B64 = """H4sIAAAAAAAC/51UsdLDIAje8yR6/3UA5yzhzrHJ3uvk2vefC1gTYyTNX9pwCskHfKCUICVHnh+kx3R7uhSSi373hOgN82P5e6JLqBBZIat2LwsYaI21HD7h6OQmjUIv36qg6swlEnCNRCM54ihQRQFx7feTvtGUBdF3rQPNk5sVRAQUXlerqSqRCxp/r4iEolAlG8i3+5ECiVRl392WXq6tgw+qLNdAr71zyr06voysFqlTBP26qKBYXGbhOs2Gtcs2wWdm7QzPfDK90MGNdQq5/iYhjJ96UUdZBP22KANTVf/vRpy5vk62UGMmTgtUFPIXhVtMMAvQz+dQIAkZkX+iO8MkpHU5s9tUDxQILGnW/L8D7cf98lE+5oUjo5abxJezNRodvpJtM0Y6DCuChXrsl2EtlRzP/GUSTu7jbZ+ZVrM1UVaGeil1WNiTcB3wjIg3Yd+jPskGAAA="""


def unpack(value):
    return gzip.decompress(base64.b64decode(value)).decode("utf-8").splitlines()


train_order = unpack(TRAIN_B64)
validation_smiles = unpack(VALIDATION_B64)

FRACTIONS = {"25": 42, "100": 166}
TRAINING_SEEDS = [11, 22, 33]
GENERATION_SEEDS = [101, 202, 303]
N_SAMPLES = 1000
TARGET_EXPOSURES = 8000
MAX_LENGTH = 202

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("Train:", len(train_order), "| validation:", len(validation_smiles))

## Prior и конфигурации

In [ ]:
import hashlib
import importlib.metadata
import re
import subprocess
import sys
import urllib.request

from rdkit import Chem, rdBase


RESULT_DIR = Path("results/external_reinvent")
WEIGHT_DIR = Path("models/external_reinvent")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

PRIOR_PATH = WEIGHT_DIR / "reinvent_pubchem.prior"
PRIOR_URL = "https://zenodo.org/records/20701824/files/reinvent_pubchem.prior?download=1"
RANDOM_SMILES_PER_MOLECULE = 5
TEMPERATURE = 1.0

if not PRIOR_PATH.exists():
    urllib.request.urlretrieve(PRIOR_URL, PRIOR_PATH)

prior_data = torch.load(PRIOR_PATH, map_location="cpu", weights_only=False)
prior_tokens = set(prior_data["vocabulary"]["tokens"])
prior_tokenizer = prior_data["tokenizer"]


def unknown_tokens(smiles):
    tokens = prior_tokenizer.tokenize(str(smiles), with_begin_and_end=False)
    return sorted(set(tokens) - prior_tokens)


def randomized_smiles(smiles_values, seed):
    rdBase.SeedRandomNumberGenerator(seed)
    rows = []
    for smiles in smiles_values:
        mol = Chem.MolFromSmiles(smiles)
        for _ in range(RANDOM_SMILES_PER_MOLECULE):
            rows.append(Chem.MolToSmiles(mol, canonical=False, doRandom=True))
    return rows


def write_smiles(path, values):
    pd.Series(values).to_csv(path, index=False, header=False)


def make_tl_config(path, model_path, train_file, validation_file, epochs, training_seed):
    text = f'''run_type = "transfer_learning"
device = "{"cuda:0" if torch.cuda.is_available() else "cpu"}"
tb_logdir = "{(RESULT_DIR / f"tb_{training_seed}").as_posix()}"

[parameters]
num_epochs = {epochs}
save_every_n_epochs = 1
batch_size = 50
shuffle_each_epoch = false
num_refs = 0
sample_batch_size = 100
input_model_file = "{PRIOR_PATH.as_posix()}"
smiles_file = "{train_file.as_posix()}"
validation_smiles_file = "{validation_file.as_posix()}"
output_model_file = "{model_path.as_posix()}"
standardize_smiles = false
randomize_smiles = false
isomeric_smiles = true
max_sequence_length = 202
clip_gradient_norm = 1.0
'''
    path.write_text(text, encoding="utf-8")


def make_sampling_config(path, model_path, output_path):
    text = f'''run_type = "sampling"
device = "{"cuda:0" if torch.cuda.is_available() else "cpu"}"

[parameters]
model_file = "{model_path.as_posix()}"
output_file = "{output_path.as_posix()}"
num_smiles = {N_SAMPLES}
unique_molecules = false
randomize_smiles = false
isomeric_smiles = true
temperature = {TEMPERATURE}
'''
    path.write_text(text, encoding="utf-8")


def run_reinvent(config_path, seed, log_path):
    subprocess.run(
        ["reinvent", "-l", str(log_path), "-s", str(seed), str(config_path)],
        check=True,
    )


def best_checkpoint(log_path, model_path):
    text = log_path.read_text(encoding="utf-8")
    match = re.search(r"Best validation loss \(([-0-9.]+)\) was at epoch (\d+)", text)
    best_loss = float(match.group(1))
    best_epoch = int(match.group(2))
    return Path(f"{model_path}.{best_epoch}.chkpt"), best_epoch, best_loss


print("REINVENT:", importlib.metadata.version("reinvent"))
print("mmpdb:", importlib.metadata.version("mmpdb"))
print("Prior MD5:", hashlib.md5(PRIOR_PATH.read_bytes()).hexdigest())

## Запуск

In [ ]:
coverage = pd.DataFrame({"smiles": train_order})
coverage["unknown_tokens"] = coverage["smiles"].map(unknown_tokens)
coverage["supported"] = ~coverage["unknown_tokens"].map(bool)
coverage.to_csv(RESULT_DIR / "representation_coverage.csv", index=False)

supported_validation = [s for s in validation_smiles if not unknown_tokens(s)]
validation_file = RESULT_DIR / "validation.smi"
write_smiles(validation_file, supported_validation)

print("Поддерживается train:", int(coverage["supported"].sum()), "из", len(coverage))
print("Поддерживается validation:", len(supported_validation), "из", len(validation_smiles))

# Zero-shot prior.
for generation_seed in GENERATION_SEEDS:
    sample_file = RESULT_DIR / f"pretrained_generation_{generation_seed}.csv"
    config_file = RESULT_DIR / f"pretrained_generation_{generation_seed}.toml"
    make_sampling_config(config_file, PRIOR_PATH, sample_file)
    run_reinvent(
        config_file,
        generation_seed,
        RESULT_DIR / f"pretrained_generation_{generation_seed}.log",
    )

checkpoint_rows = []
checkpoint_paths = []

for fraction, size in FRACTIONS.items():
    molecules = [s for s in train_order[:size] if not unknown_tokens(s)]
    randomized = randomized_smiles(molecules, 42)
    train_rows = [s for s in randomized if not unknown_tokens(s)]
    epochs = math.ceil(TARGET_EXPOSURES / len(train_rows))

    print()
    print("Доля:", fraction, "| молекул:", len(molecules), "| SMILES:", len(train_rows), "| эпох:", epochs)

    for training_seed in TRAINING_SEEDS:
        train_file = RESULT_DIR / f"fraction_{fraction}_train_seed_{training_seed}.smi"
        ordered = pd.Series(train_rows).sample(frac=1, random_state=training_seed).tolist()
        write_smiles(train_file, ordered)

        model_path = WEIGHT_DIR / f"fraction_{fraction}_seed_{training_seed}.model"
        config_file = RESULT_DIR / f"fraction_{fraction}_seed_{training_seed}.toml"
        log_file = RESULT_DIR / f"fraction_{fraction}_seed_{training_seed}.log"
        make_tl_config(
            config_file,
            model_path,
            train_file,
            validation_file,
            epochs,
            training_seed,
        )
        run_reinvent(config_file, training_seed, log_file)
        selected, best_epoch, best_loss = best_checkpoint(log_file, model_path)
        checkpoint_paths.append(selected)
        checkpoint_rows.append({
            "fraction": fraction,
            "training_seed": training_seed,
            "molecules": len(molecules),
            "training_smiles": len(train_rows),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "best_validation_loss": best_loss,
            "checkpoint": selected.name,
        })

        for generation_seed in GENERATION_SEEDS:
            sample_file = RESULT_DIR / f"fraction_{fraction}_train_{training_seed}_generation_{generation_seed}.csv"
            sample_config = RESULT_DIR / f"sample_{fraction}_{training_seed}_{generation_seed}.toml"
            make_sampling_config(sample_config, selected, sample_file)
            run_reinvent(
                sample_config,
                generation_seed,
                RESULT_DIR / f"sample_{fraction}_{training_seed}_{generation_seed}.log",
            )

checkpoints = pd.DataFrame(checkpoint_rows)
checkpoints.to_csv(RESULT_DIR / "checkpoints.csv", index=False)
print(checkpoints.round(4).to_string(index=False))

## Архивы

In [ ]:
results_archive = shutil.make_archive("external_reinvent_results", "zip", RESULT_DIR)
weights_archive = Path("external_reinvent_weights.zip")
with ZipFile(weights_archive, "w") as archive:
    archive.write(PRIOR_PATH, arcname=PRIOR_PATH.name)
    for path in checkpoint_paths:
        archive.write(path, arcname=path.name)

print("Результаты:", Path(results_archive).resolve())
print("Веса:", weights_archive.resolve())

try:
    from google.colab import files
    files.download(results_archive)
    files.download(weights_archive)
except ImportError:
    print("В Kaggle архивы находятся в разделе Output.")